#### The SparkConnect service

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.connect.session as _cs

# Clear any stale session after a server restart
_cs.SparkSession._active_spark_session = None

connect_url = "sc://localhost:15002"
spark = SparkSession.builder.remote(connect_url).appName("NotebookSparkConnectTest").getOrCreate()

print("Spark Connect OK")
print("Spark version:", spark.version)

dbs = [row.namespace for row in spark.sql("SHOW DATABASES").collect()]
print(f"\nDatabases ({len(dbs)}): {dbs}\n")

for db in dbs:
    tables = spark.sql(f"SHOW TABLES IN {db}").collect()
    print(f"[{db}] — {len(tables)} table(s)")
    for t in tables:
        print(f"  - {t.tableName} (temporary={t.isTemporary})")

In [ ]:
# Register databases/tables that exist in the warehouse but are missing from the Hive metastore.
# Run this once when the metastore has been re-created or migrated.

from pyhive import hive
import os

WAREHOUSE = "/Users/mark/Workspace/Projects/data-analysis/data/spark/warehouse"

conn = hive.Connection(host="localhost", port=10000, auth="NONE")
cur = conn.cursor()

# Find all .db directories in the warehouse that aren't 'default'
for entry in sorted(os.listdir(WAREHOUSE)):
    if not entry.endswith(".db") or entry == "default.db":
        continue
    db_name = entry[:-3]
    db_path = f"file://{WAREHOUSE}/{entry}"

    cur.execute(f"SHOW DATABASES LIKE '{db_name}'")
    if cur.fetchone():
        print(f"[{db_name}] already registered — skipping")
        continue

    cur.execute(f"CREATE DATABASE {db_name} LOCATION '{db_path}'")
    print(f"[{db_name}] database created → {db_path}")

    # Register each Parquet table in the database
    table_dir = f"{WAREHOUSE}/{entry}"
    for tbl in sorted(os.listdir(table_dir)):
        tbl_path = f"{table_dir}/{tbl}"
        if not os.path.isdir(tbl_path) or tbl.startswith("_"):
            continue
        cur.execute(f"CREATE TABLE IF NOT EXISTS {db_name}.{tbl} USING PARQUET LOCATION 'file://{tbl_path}'")
        print(f"  [{db_name}.{tbl}] table registered → {tbl_path}")

cur.close()
conn.close()
print("\nDone.")

In [ ]:
from pyhive import hive

conn = hive.Connection(host="localhost", port=10000, auth="NONE")
cursor = conn.cursor()

cursor.execute("SHOW DATABASES")
dbs_thrift = [row[0] for row in cursor.fetchall()]
print(f"Thrift Server — Databases ({len(dbs_thrift)}): {dbs_thrift}\n")

for db in dbs_thrift:
    cursor.execute(f"SHOW TABLES IN {db}")
    tables = cursor.fetchall()
    print(f"[{db}] — {len(tables)} table(s)")
    for t in tables:
        print(f"  - {t[1]}")

cursor.close()
conn.close()

In [3]:
from pyspark.sql import SparkSession
import pyspark.sql.connect.session as _cs
from pyspark.sql import functions as F

# Clear any stale session after a server restart
_cs.SparkSession._active_spark_session = None

connect_url = "sc://localhost:15002"
spark = SparkSession.builder.remote(connect_url).appName("NotebookSparkConnectTest").getOrCreate()

# Set this manually to control the namespace date.
business_date = "2026-04-24"  # <- change this value
namespace_db = f"data_{business_date.replace('-', '')}"

# Load dx from the selected business-date namespace.
dx_with_random_id = (
    spark.table(f"{namespace_db}.dx")
    .withColumn("random_id", (F.rand() * 1_000_000_000).cast("bigint"))
)

# Save back as dx_modified in the same namespace.
spark.sql(f"CREATE DATABASE IF NOT EXISTS {namespace_db}")
dx_with_random_id.write.mode("overwrite").saveAsTable(f"{namespace_db}.dx_modified")

print(f"Namespace: {namespace_db}")
print(f"Saved table: {namespace_db}.dx_modified")
dx_with_random_id.show(20, truncate=False)

Namespace: data_20260424
Saved table: data_20260424.dx_modified
+--------+------+-----------+-----------+---------+----------+---------+
|type    |app_id|uid        |uid_version|account  |as_of_date|random_id|
+--------+------+-----------+-----------+---------+----------+---------+
|POSITION|APP001|UID00150001|2          |ACC000002|1998      |176777356|
|POSITION|APP001|UID00150002|3          |ACC000003|1998      |990949561|
|POSITION|APP001|UID00150003|4          |ACC000004|1998      |107655433|
|POSITION|APP001|UID00150004|5          |ACC000005|1998      |685240845|
|POSITION|APP001|UID00150005|1          |ACC000006|1998      |660226150|
|POSITION|APP001|UID00150006|2          |ACC000007|1998      |837578615|
|POSITION|APP001|UID00150007|3          |ACC000008|1998      |638230707|
|POSITION|APP001|UID00150008|4          |ACC000009|1998      |899910479|
|POSITION|APP001|UID00150009|5          |ACC000010|1998      |710534781|
|POSITION|APP001|UID00150010|1          |ACC000011|1998     